# Tesis RoBERTuito en Colab

Abre este cuaderno desde la carpeta **Tesis_RoBERTuito_M0_M1_M2** de Mi unidad y pulsa **Ejecutar todo**. Autoriza el acceso a Drive cuando Colab lo solicite. Las tablas y evaluaciones quedan en una carpeta con fecha.

Los cuadernos 00–09 de Drive contienen sus salidas históricas para consultarlas directamente. Este iniciador crea las salidas de cada ejecución nueva.

**Revisión docente.** Con `ENTRENAR_DESDE_INICIO = False`, Ejecutar todo reutiliza los pesos, las propuestas y las predicciones de evaluación guardadas. Las tablas y figuras se regeneran a partir de esos archivos.

In [ ]:
# @title 1. Conectar Drive y preparar la carpeta de trabajo
from datetime import datetime, timezone
import json
from pathlib import Path
from shutil import copyfile, copytree
import subprocess
import sys
from google.colab import drive

ENTRENAR_DESDE_INICIO = False  # @param {type:"boolean"}
drive.mount("/content/drive")
carpeta_drive = Path("/content/drive/MyDrive") / "Tesis_RoBERTuito_M0_M1_M2"
if not (carpeta_drive / "corrida.json").is_file():
    raise FileNotFoundError("Ubica la carpeta Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

identificador = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
proyecto = Path("/content") / f"tesis_{identificador}"
proyecto.mkdir()
for nombre in ("data", "insumos", "TABLAS", "evaluacion"):
    copytree(carpeta_drive / nombre, proyecto / nombre)
for p in carpeta_drive.glob("0[0-9]_*.ipynb"):
    copyfile(p, proyecto / p.name)
for nombre in ("entrenamiento.py", "evaluar.py", "rutas.py",
               "ejecutar_cuadernos.py", "gestion_modelos.py", "requirements-cuadernos.txt"):
    copyfile(carpeta_drive / nombre, proyecto / nombre)

base = "corrida_03_epocas"
config = json.loads((carpeta_drive / "corrida.json").read_text(encoding="utf-8"))
corrida = f"colab_{identificador}" if ENTRENAR_DESDE_INICIO else config["corrida"]
carpeta_base = proyecto / "modelos" / base
copytree(carpeta_drive / "modelos" / base / "M0", carpeta_base / "M0")
if not ENTRENAR_DESDE_INICIO:
    for modelo in ("M0", "M1", "M2"):
        destino = proyecto / "modelos" / corrida / modelo
        if not destino.exists():
            copytree(carpeta_drive / "modelos" / corrida / modelo, destino)
(proyecto / "corrida.json").write_text(json.dumps({"corrida": corrida}), encoding="utf-8")
print("Corrida seleccionada:", corrida)

In [ ]:
# @title 2. Instalar dependencias y ejecutar los cuadernos 00–09
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(proyecto / "requirements-cuadernos.txt")], check=True)
if ENTRENAR_DESDE_INICIO:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Selecciona un entorno de ejecución con GPU para entrenar.")

sys.path.insert(0, str(proyecto))
from gestion_modelos import preguntar_modelo
respuestas = ["no", "no", "no"] if ENTRENAR_DESDE_INICIO else ["si", "si", "si"]

salida = proyecto / "salidas" / f"colab_{identificador}"
comando = [sys.executable, "-u", "ejecutar_cuadernos.py", "--salida", str(salida), "--respuestas", *respuestas]
if ENTRENAR_DESDE_INICIO:
    comando.append("--entrenar")
subprocess.run(comando, cwd=proyecto, check=True)

In [ ]:
# @title 3. Guardar los resultados en Drive
destino = carpeta_drive / "salidas" / salida.name
destino.parent.mkdir(exist_ok=True)
copytree(salida, destino, dirs_exist_ok=True)
config_final = json.loads((proyecto / "corrida.json").read_text(encoding="utf-8"))
corrida_final = config_final["corrida"]
if ENTRENAR_DESDE_INICIO or corrida_final != config["corrida"]:
    copytree(proyecto / "modelos" / corrida_final, carpeta_drive / "modelos" / corrida_final, dirs_exist_ok=True)
    (carpeta_drive / "corrida.json").write_text(json.dumps(config_final), encoding="utf-8")
print("Resultados disponibles en Mi unidad /", destino.relative_to(carpeta_drive.parent))